# 02_Data_Cleaning

In [2]:
from pathlib import Path
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)


PROJECT_ROOT = Path("..")
RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA.mkdir(
    parents=True,
    exist_ok=True,
)


pd.set_option(
    "display.max_columns",
    None,
)

## Utility Functions

Load, clean, validate and save datasets.

In [3]:
def load_csv(path):
    return pd.read_csv(path)

def save_csv(df,name):
    df.to_csv(PROCESSED_DATA/f'{name}.csv',index=False)

def convert_datetime(df,cols):
    for col in cols:
        if col in df.columns:
            df[col]=pd.to_datetime(df[col],errors='coerce')
    return df

def trim(df):
    for col in df.select_dtypes(include='object'):
        df[col]=df[col].astype(str).str.strip()
    return df

def optimize(df):
    for col in df.select_dtypes(include='float'):
        df[col]=pd.to_numeric(df[col],downcast='float')
    for col in df.select_dtypes(include='integer'):
        df[col]=pd.to_numeric(df[col],downcast='integer')
    return df

def fk_check(child,parent,key):
    return child[key].isin(parent[key]).all() if key in child and key in parent else None

## Load Datasets

In [6]:
raw_files=sorted(RAW_DATA.glob('*.csv'))
datasets={p.stem:load_csv(p) for p in raw_files}
print(datasets.keys())

dict_keys(['customers', 'geolocation', 'order_items', 'order_payments', 'order_reviews', 'orders', 'products', 'sellers'])


## Schema Validation

In [7]:
for n,df in datasets.items():
    print('\n',n,df.shape)
display(df.dtypes.to_frame('dtype'))
display(df.isna().sum().to_frame('missing'))
print('Duplicates',df.duplicated().sum())


 customers (1000000, 9)

 geolocation (11500, 5)

 order_items (2199819, 8)

 order_payments (1149371, 5)

 order_reviews (933748, 7)

 orders (1000000, 8)

 products (2000, 10)

 sellers (500, 8)


,dtype
seller_id,object
seller_company_name,object
seller_contact_name,object
seller_contact_gender,object
seller_contact_age,int64
seller_zip_code_prefix,int64
seller_city,object
seller_state,object


,missing
seller_id,0
seller_company_name,0
seller_contact_name,0
seller_contact_gender,0
seller_contact_age,0
seller_zip_code_prefix,0
seller_city,0
seller_state,0


Duplicates 0


## Datetime Conversion

In [8]:
datetime_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "order_items": [
        "shipping_limit_date",
    ],
    "order_reviews": [
        "review_creation_date",
        "review_answer_timestamp",
    ],
}


for dataset_name, columns in datetime_columns.items():

    if dataset_name in datasets:

        datasets[dataset_name] = convert_datetime(
            datasets[dataset_name],
            columns,
        )

## Missing Value Analysis

In [9]:
if "orders" in datasets:

    missing_values = (
        datasets["orders"][
            [
                "order_status",
                "order_delivered_carrier_date",
                "order_delivered_customer_date",
            ]].isna().sum().to_frame("Missing Values"))

    display(missing_values)

,Missing Values
order_status,0
order_delivered_carrier_date,66252
order_delivered_customer_date,66252


## Cleaning

In [10]:
for k in datasets: datasets[k]=optimize(trim(datasets[k]))

## Outlier Detection

In [11]:
for n,df in datasets.items():
 nums=df.select_dtypes(include='number')
 if not nums.empty:
  display(nums.describe().T)

,count,mean,std,min,25%,50%,75%,max
customer_age,1000000.0,35.383124,11.219629,18.0,27.0,35.0,43.0,70.0
customer_zip_code_prefix,1000000.0,57314.170948,30624.162507,10010.0,30369.0,77012.0,90019.0,98199.0


,count,mean,std,min,25%,50%,75%,max
zip_code_prefix,11500.0,58391.656000,31888.147019,10010.000000,30378.000000,77022.000000,90024.000000,98199.000000
geolocation_lat,11500.0,35.503990,6.133895,25.661831,29.816575,34.071735,40.699645,47.705940
geolocation_lng,11500.0,-95.521133,17.745214,-122.431992,-118.172159,-95.300884,-80.147593,-73.906059


,count,mean,std,min,25%,50%,75%,max
order_item_id,2199819.0,1.847999,0.972301,1.00,1.000000,2.000000,2.000000,6.000000
price,2199819.0,446.918152,539.856995,10.01,64.089996,176.589996,736.590027,1999.849976
freight_value,2199819.0,108.665977,63.878536,5.06,64.589996,105.669998,159.369995,250.289993
discount_rate,2199819.0,0.017238,0.058647,0.00,0.000000,0.000000,0.000000,0.300000


,count,mean,std,min,25%,50%,75%,max
payment_sequential,1149371.0,1.129959,0.336258,1.00,1.000000,1.000000,1.000000,2.000000
payment_installments,1149371.0,3.949850,3.732548,1.00,1.000000,1.000000,7.000000,12.000000
payment_value,1149371.0,1063.350830,963.745422,15.55,325.829987,745.390015,1560.079956,11054.519531


,count,mean,std,min,25%,50%,75%,max
review_score,933748.0,4.034775,1.22905,1.0,4.0,5.0,5.0,5.0


,count,mean,std,min,25%,50%,75%,max
product_weight_g,2000.0,6555.015000,10777.216718,100.00,608.750000,1424.000000,6175.500000,49869.000000
product_length_cm,2000.0,48.938500,43.596801,5.00,19.000000,33.000000,62.000000,199.000000
product_height_cm,2000.0,16.744500,7.294095,5.00,10.000000,17.000000,23.000000,29.000000
product_width_cm,2000.0,16.810000,7.085337,5.00,11.000000,17.000000,23.000000,29.000000
cost,2000.0,223.647324,339.807831,4.43,23.552500,73.469997,275.747490,1693.119995
price,2000.0,322.651581,419.820251,10.01,45.487499,150.924995,404.587486,1999.849976


,count,mean,std,min,25%,50%,75%,max
seller_contact_age,500.0,35.938,11.363116,18.0,27.0,36.0,44.00,70.0
seller_zip_code_prefix,500.0,58796.928,31977.469657,10011.0,30395.0,77015.0,90031.25,98198.0


## Referential Integrity

In [12]:
relationship_checks = []


if {"customers", "orders"}.issubset(datasets):

    relationship_checks.append(["orders.customer_id", fk_check( datasets["orders"], datasets["customers"], "customer_id",),])


if {"orders", "order_items"}.issubset(datasets):

    relationship_checks.append(
        [
            "order_items.order_id",
            fk_check(
                datasets["order_items"],
                datasets["orders"],
                "order_id",
            ),
        ]
    )


relationship_summary = pd.DataFrame(
    relationship_checks,
    columns=["Relationship", "Valid",],)

display(relationship_summary)

,Relationship,Valid
0,orders.customer_id,True
1,order_items.order_id,True


## Master Dataset

In [13]:
master_df=datasets['orders'].merge(datasets['customers'],on='customer_id',how='left')
master_df=master_df.merge(datasets['order_items'],on='order_id',how='left')
master_df=master_df.merge(datasets['products'],on='product_id',how='left')
master_df=master_df.merge(datasets['sellers'],on='seller_id',how='left')
master_df=master_df.merge(datasets['order_payments'],on='order_id',how='left')
master_df=master_df.merge(datasets['order_reviews'],on='order_id',how='left')
display(master_df.head())
print(master_df.shape)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_name,customer_gender,customer_age,customer_zip_code_prefix,customer_city,customer_state,customer_segment,order_item_id,product_id,seller_id,shipping_limit_date,price_x,freight_value,discount_rate,product_category_name,product_name,product_brand,product_weight_g,product_length_cm,product_height_cm,product_width_cm,cost,price_y,seller_company_name,seller_contact_name,seller_contact_gender,seller_contact_age,seller_zip_code_prefix,seller_city,seller_state,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,720fd9e9-cdeb-4dec-a187-f71586eb085a,1e2e2881-a0eb-4cb0-829f-a566e810d05f,canceled,2025-12-27 07:07:20,2025-12-27 08:33:20,NaT,NaT,2026-01-04 08:33:20,d45cdff2-5195-41e2-a0e5-6fe597e378dd,James Ramos,M,47,85037,Phoenix,AZ,Consumer,1,89804d82-33a1-4558-8fa2-9b0252a2a406,36c2b043-ccf4-4d9c-a0bb-4b8d28737fd7,2025-12-30 07:07:20,916.900024,45.470001,0.0,furniture,Nova Desk - Industrial,Nova,24130,108,10,13,395.239990,916.900024,Jones-Russell,Nicholas Thompson,M,18,90086,Los Angeles,CA,1,paypal,1,2084.010010,NaN,NaN,NaN,NaN,NaT,NaT
1,720fd9e9-cdeb-4dec-a187-f71586eb085a,1e2e2881-a0eb-4cb0-829f-a566e810d05f,canceled,2025-12-27 07:07:20,2025-12-27 08:33:20,NaT,NaT,2026-01-04 08:33:20,d45cdff2-5195-41e2-a0e5-6fe597e378dd,James Ramos,M,47,85037,Phoenix,AZ,Consumer,2,ac10b78c-342e-4b50-9ea9-bddde7db2e79,09936504-415b-4f90-a5b8-ad15a5be67d0,2025-12-30 07:07:20,817.140015,172.479996,0.0,electronics,Vertex Smartwatch S468,Vertex,586,28,19,23,717.809998,817.140015,Crawford-Pugh,Jeffrey Stone,M,70,19137,Philadelphia,PA,1,paypal,1,2084.010010,NaN,NaN,NaN,NaN,NaT,NaT
2,720fd9e9-cdeb-4dec-a187-f71586eb085a,1e2e2881-a0eb-4cb0-829f-a566e810d05f,canceled,2025-12-27 07:07:20,2025-12-27 08:33:20,NaT,NaT,2026-01-04 08:33:20,d45cdff2-5195-41e2-a0e5-6fe597e378dd,James Ramos,M,47,85037,Phoenix,AZ,Consumer,3,f18e6310-0e75-4b7f-bc20-90ac1e7bd466,7d336012-7101-4a66-b6bc-c269620d8df9,2025-12-30 07:07:20,37.040001,94.980003,0.0,fashion,Crest Minimal Dress,Crest,697,55,7,7,9.680000,37.040001,Fields PLC,Jill Williams,F,27,98129,Seattle,WA,1,paypal,1,2084.010010,NaN,NaN,NaN,NaN,NaT,NaT
3,c0142972-63fa-4af2-8070-f583ab769847,380b7418-308c-4bf7-b2bd-3ee446cb9ea6,delivered,2019-06-07 19:30:44,2019-06-08 05:08:44,2019-06-09 05:08:44,2019-06-14 05:08:44,2019-06-16 05:08:44,35cee471-325e-4ad2-8e4e-7b169dc6df81,Brian Jackson,M,18,90060,Los Angeles,CA,Consumer,1,5816f107-b725-4c41-b794-298bf9669a41,91fe2cc8-51a5-4c79-9c12-cea5ae013f55,2019-06-10 19:30:44,917.979980,27.420000,0.0,furniture,Zenith Oak Coffee Table,Zenith,43065,143,27,23,587.820007,917.979980,Johnson-Garcia,Michelle Gentry,F,19,90021,Los Angeles,CA,1,credit_card,6,1406.069946,0ba49642-47bb-4817-9cc2-bb9ded7de7c5,5.0,Perfect,The quality is amazing. Delivery was quick.,2019-06-19 05:08:44,2019-06-19 07:08:44
4,c0142972-63fa-4af2-8070-f583ab769847,380b7418-308c-4bf7-b2bd-3ee446cb9ea6,delivered,2019-06-07 19:30:44,2019-06-08 05:08:44,2019-06-09 05:08:44,2019-06-14 05:08:44,2019-06-16 05:08:44,35cee471-325e-4ad2-8e4e-7b169dc6df81,Brian Jackson,M,18,90060,Los Angeles,CA,Consumer,2,f4b3ce27-568d-4c33-9248-6c314635f80a,54165da9-b969-4d50-9b61-ffd0d8b4d731,2019-06-10 19:30:44,28.709999,145.250000,0.0,toys,Nimbus Doll Toy,Nimbus,788,18,27,10,15.450000,28.709999,Alvarez-Phillips,Tyler Johnson,M,34,60660,Chicago,IL,1,credit_card,6,1406.069946,0ba49642-47bb-4817-9cc2-bb9ded7de7c5,5.0,Perfect,The quality is amazing. Delivery was quick.,2019-06-19 05:08:44,2019-06-19 07:08:44


(2530433, 49)


## Save Outputs

In [14]:
for n,df in datasets.items(): save_csv(df,n)
save_csv(master_df,'master_df')
print('Saved processed datasets.')

Saved processed datasets.


## Summary

The data cleaning process successfully validated and integrated the datasets for downstream analysis. Missing value analysis identified that only the **delivery-related columns** in the `orders` dataset contain null values, with **66,252 missing records** each in `order_delivered_carrier_date` and `order_delivered_customer_date`. These missing values are likely associated with undelivered or cancelled orders and should be retained unless business rules indicate otherwise.

Relationship validation confirmed that the primary and foreign key relationships are consistent across the datasets. All `customer_id` values in the `orders` table exist in the `customers` table, and all `order_id` values in the `order_items` table exist in the `orders` table, indicating no orphan records or referential integrity issues.

Following validation, the datasets were successfully merged into a consolidated **master dataset** using `orders` as the central fact table. The final integrated dataset contains **2,530,433 rows** and **49 columns**, combining customer, order, product, seller, payment, and review information into a single analytical table.

Finally, all cleaned individual datasets, along with the consolidated `master_df`, were saved to the processed data directory. The data is now prepared for Exploratory Data Analysis (EDA), feature engineering, and machine learning model development.

---

# Next Steps

- Verify the merged dataset by checking its shape, data types, and missing values.
- Validate that the merge has not introduced duplicate or inconsistent records.
- Perform univariate, bivariate, and multivariate Exploratory Data Analysis (EDA).
- Analyze customer purchasing behavior, product performance, seller performance, and payment trends.
- Engineer new business features such as Recency, Frequency, Monetary (RFM), Average Order Value (AOV), Customer Lifetime Value (CLV), and delivery duration.
- Save the finalized analytical dataset for machine learning model development.